# 5.5 Project: Multi-Agent Research Team

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

**Prerequisites:** all of Modules 0 to 4, and lessons 5.1 to 5.4 (persistence, human-in-the-loop, supervisor/subgraph patterns, parallel execution and map-reduce). The skeleton below runs with no API key. The LLM-backed steps are marked and need an `OPENAI_API_KEY` (or Ollama as the open-weight alternative).

## Learning objectives

- Compare three ways to organise multiple agents, supervisor, pipeline, and handoffs, and choose between them for a concrete job.
- Decide what lives in shared state versus per-agent state, and where persistence and human-in-the-loop approval slots slot into a multi-agent run.
- Build a planner, researcher (parallel, map-reduce over sources), writer, and critic team on LangGraph using `Send` fan-out and a critic loop with a bounded revision budget.
- Evaluate the final report for faithfulness against its sources with a quote-match check, and read the result table honestly.
- Ship the project like a professional: working code, a sample report with citations, an eval table, and a README.

## Why a research team

A single agent given a research brief does three jobs badly at once: it plans shallowly, it reads sources serially (slow and context-heavy), and it grades its own homework. Splitting the jobs across specialised agents fixes each one in a different way. The planner decomposes the brief without committing to any source. The researchers work in parallel, so five sources take roughly as long as one. The writer has one job, synthesis, and sees only distilled findings, not raw documents. The critic has one job too, checking the report against the evidence, and it is allowed to send the writer back to work, but only a bounded number of times.

That last detail is the whole project in miniature. A critic loop without a budget is a while-loop with a model in the condition. It will, sooner or later, not converge, and you will pay for every lap.

## Architecture: three shapes, one choice

Before you write a line, pick the control shape. There are three, and the research team is a good excuse to learn all of them.

**Pipeline.** Planner to researchers to writer to critic, in a fixed order. Deterministic, easy to debug, easy to test. The cost is rigidity: the critic can reject, but it cannot ask the planner for a new research question, because there is no edge back there.

**Supervisor.** One agent in the middle decides, at each step, which specialist to call next. Flexible: the supervisor can re-plan, call more researchers, or skip the critic when the draft is obviously fine. The cost is the supervisor itself. Every routing decision is a model call, routing mistakes are silent, and the supervisor's context grows with every specialist's output.

**Handoffs.** Each agent decides who goes next, usually by naming the next agent in its output. This is the shape agent SDKs push hardest. The cost is control: the workflow is whatever the models decided today, which is hard to test and hard to bound.

This project uses a **pipeline with one controlled loop**: planner, parallel researchers, writer, critic, and the critic routes back to the writer at most `MAX_REVISIONS` times. The pipeline shape is deliberate. A research report has a natural fixed order, the loop is the only place where judgement is needed, and a fixed shape makes the eval meaningful: you can compare runs because every run has the same steps. The supervisor and handoff shapes are discussed in 5.3; your README should say why you did not use them here. "The workflow has a natural fixed order and exactly one decision point" is a complete justification. That sentence is worth marks.

## Shared state versus per-agent state

The team shares one graph state, but not every field is shared in the same way:

- `question` (string): read by everyone, written by no node after the start. Treat it as immutable input.
- `plan` (list of sub-questions): written once by the planner, read by the fan-out edge. Immutable after planning.
- `findings` (list of dicts, with an `operator.add` reducer): the only field every researcher writes. The reducer is what makes parallel writes safe. Never let a node overwrite this field; always append.
- `draft` (string): written by the writer, read by the critic, overwritten on revision. This is the loop's working memory.
- `revisions` (int): the critic loop's budget counter. Written by the writer node, read by the gate.

A useful rule: **inputs are immutable, working sets are append-only, the draft is the only mutable field, and the budget is a number.** When a multi-agent system misbehaves, the first thing to check is which node wrote to a field it should only have read.

Each researcher also gets **per-agent input** through `Send`: the sub-question and the path to its source file. The researcher never sees the full plan or the other researchers' findings. This is not just tidy design. It is the fix for researcher context overload (see "What can go wrong"): each researcher carries one question and one document, and the writer never sees raw documents at all.

## Where persistence and approval slot in

Compile the graph with a checkpointer (5.1). One thread id per research brief. That gives you resume-after-crash and the time-travel debugger for free, which matters the moment a 20-minute research run dies at the writer.

The human-in-the-loop approval (5.2) goes in exactly one place: **after the critic accepts the draft and before the report is "published"** (written to disk or sent anywhere). Research is read-only, so the research stages need no approval. Publication is the irreversible step, so that is where `interrupt()` lives. Do not put an approval inside the researcher loop. Approving five source reads is busywork, and busywork approvals get rubber-stamped, which defeats the point.

## Build spec

The spec below is a complete skeleton. Steps 1 to 4 run with no API key. Step 5 marks the LLM upgrades. Every stub is honest about what it stands in for.

### Step 0: the document set

Create a `docs/` folder next to the script with 3 to 6 markdown files on one topic. Real documents, not lorem ipsum: changelogs, product docs, course notes, anything you have the right to use. The whole project reads from these files, and the eval checks the report against them, so pick a topic you can actually verify.

### Step 1: state and the planner

In [ ]:
import operator
import pathlib
import re
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send, interrupt

DOCS_DIR = pathlib.Path("docs")
MAX_RESEARCHERS = 5    # cap the fan-out: one researcher per source file, at most
MAX_REVISIONS = 2      # cap the critic loop: the budget is a number, not a hope


class ResearchState(TypedDict, total=False):
    question: str
    plan: list                      # sub-questions, one per source file
    findings: Annotated[list, operator.add]   # append-only; each entry is a dict
    draft: str
    revisions: int


def planner(state: ResearchState) -> dict:
    """STUB (no LLM). One research question per source file, capped.
    LLM version: ask the model to decompose state['question'] into
    focused sub-questions; keep the cap."""
    files = sorted(DOCS_DIR.glob("*.md"))[:MAX_RESEARCHERS]
    topic = state["question"]
    return {
        "plan": [
            {"sub_question": f"What does {f.stem} say about {topic}?", "source": str(f)}
            for f in files
        ]
    }

### Step 2: researcher fan-out with `Send`

The conditional edge returns one `Send` per sub-question. Each researcher node receives only its own dict, not the whole state. This is the map half of map-reduce (5.4), and the reducer that merges the findings is the reduce half.

In [ ]:
def fan_out(state: ResearchState) -> list[Send]:
    plan = state.get("plan", [])
    return [Send("researcher", item) for item in plan[:MAX_RESEARCHERS]]


def researcher(item: dict) -> dict:
    """STUB (no LLM). Keyword scan of one file; returns quotes, not summaries.
    LLM version: the model reads the file and extracts claims with verbatim
    quotes. The contract stays the same: findings cite exact source text."""
    text = pathlib.Path(item["source"]).read_text(encoding="utf-8")
    topic_words = [w.lower() for w in item["sub_question"].split() if len(w) > 4]
    quotes = []
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        if any(w in sentence.lower() for w in topic_words):
            quotes.append(sentence.strip())
        if len(quotes) == 4:
            break
    return {
        "findings": [
            {
                "source": item["source"],
                "sub_question": item["sub_question"],
                "quotes": quotes,
            }
        ]
    }

Why the stub returns quotes rather than summaries: the eval in Step 6 checks the report against source text. A researcher that paraphrases breaks the audit trail. Real or stub, the researcher's output contract is **verbatim quotes plus the source path**. Teach the team to cite first and summarise later.

### Step 3: writer and the bounded critic loop

The writer turns findings into a report with citations. The critic reads the report and the findings, then routes: back to the writer while budget remains, to the approval gate when satisfied or exhausted.

In [ ]:
def writer(state: ResearchState) -> dict:
    """STUB (no LLM). Formats findings as a cited report.
    LLM version: the model synthesises the findings into prose. Give it the
    findings list only, never the raw documents (context discipline)."""
    lines = [f"# Research report: {state['question']}", ""]
    for f in state.get("findings", []):
        lines.append(f"## From {pathlib.Path(f['source']).stem}")
        for q in f["quotes"]:
            lines.append(f"- {q} [{pathlib.Path(f['source']).name}]")
        lines.append("")
    return {"draft": "\n".join(lines), "revisions": state.get("revisions", 0) + 1}


def critic_gate(state: ResearchState) -> Command:
    """The budget lives here. Reject at most MAX_REVISIONS times, then ship
    what exists. A critic that can loop forever is a billing incident."""
    issues = check_citations(state.get("draft", ""), state.get("findings", []))
    if issues and state.get("revisions", 0) < MAX_REVISIONS:
        return Command(goto="writer", update={"draft": state["draft"]})
    return Command(goto="approve")


def check_citations(draft: str, findings: list) -> list:
    """Every [file] citation must have a matching quote in the findings."""
    problems = []
    cited = set(re.findall(r"\[([^\]]+\.md)\]", draft))
    sourced = {pathlib.Path(f["source"]).name for f in findings}
    for c in cited:
        if c not in sourced:
            problems.append(f"citation [{c}] has no matching source")
    return problems

Note the routing shape: `critic_gate` returns a `Command` with `goto`, so it both inspects and routes, the pattern from 4.3. When the budget is exhausted, the gate routes forward anyway and the shipped draft carries its flaws. That is the correct behaviour. A gate that refuses to ship after the budget is spent leaves the run stuck, and "stuck" is worse than "imperfect but evaluated", because the eval table will show the imperfection.

### Step 4: approval gate and assembly

In [ ]:
def approve(state: ResearchState) -> dict:
    decision = interrupt({
        "action": "publish_research_report",
        "question": state["question"],
        "revisions_used": state.get("revisions", 0),
        "preview": state.get("draft", "")[:500],
    })
    return {"approved": bool(decision)}


builder = StateGraph(ResearchState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)
builder.add_node("critic_gate", critic_gate)
builder.add_node("approve", approve)
builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", fan_out)
builder.add_edge("researcher", "writer")   # all researchers join at the writer
builder.add_edge("writer", "critic_gate")
builder.add_edge("approve", END)

graph = builder.compile(checkpointer=InMemorySaver())

Two things to notice. First, the edge from `researcher` to `writer` is the fan-in: the writer runs once all researchers finish, and the `findings` reducer has merged their outputs by then. Second, `approve` uses `interrupt()` from 4.4, so resume works on the same thread id, and the irreversible step, writing or sending the report, happens after the resume, never before the pause.

Run it:

In [ ]:
config = {"configurable": {"thread_id": "brief-001"}}
result = graph.invoke({"question": "How does LangGraph checkpointing work"}, config)
print(result["__interrupt__"][0].value["preview"])   # the approval payload
final = graph.invoke(Command(resume=True), config)

### Step 5: the LLM upgrades (needs `OPENAI_API_KEY`, or Ollama)

Swap the stubs one at a time, keeping every contract:

- **Planner:** `ChatOpenAI(model="gpt-4.1-mini")` with a system prompt that returns a JSON list of sub-questions. Validate the JSON; if parsing fails, fall back to one question per file. A planner that crashes on bad JSON is a planner you cannot trust with a budget.
- **Researcher:** the model reads the file text and returns claims with verbatim quotes. Keep the per-agent input narrow: sub-question plus file text, nothing else.
- **Writer:** the model receives the findings list, never the raw documents.
- **Critic:** the model receives the draft plus the findings and returns a list of issues. The gate logic, budget check and routing, stays exactly as written. The model advises; the code decides.

Ollama alternative: replace `ChatOpenAI` with `ChatOllama(model="llama3.1")` from `langchain-ollama`. Same contracts, no key, slower, and the small models are noticeably worse at verbatim quoting, which is itself a useful observation for your README.

### Step 6: the evaluator (faithfulness against sources)

This is the project's exam. For every citation in the final report, verify the quoted text appears verbatim in the cited source file. Score:

- **Supported:** the quote is found in the cited file.
- **Unsupported:** a citation points at a file that never contained the quote. This is citation fabrication, the failure mode that matters.
- **Uncited claim:** a sentence in the report makes a factual claim with no citation at all. Count these separately; they are not fabrication, they are missing evidence.

In [ ]:
def evaluate_faithfulness(draft: str, findings: list) -> dict:
    rows, supported, total = [], 0, 0
    source_text = {}
    for f in findings:
        source_text.setdefault(f["source"], pathlib.Path(f["source"]).read_text(encoding="utf-8"))
    for m in re.finditer(r"-\s(.+?)\s\[([^\]]+\.md)\]", draft):
        claim, cited = m.group(1).strip(), m.group(2)
        total += 1
        hit = any(cited in pathlib.Path(s).name and claim in text
                  for s, text in source_text.items())
        supported += hit
        rows.append({"claim": claim[:80], "cited": cited,
                     "verdict": "supported" if hit else "UNSUPPORTED"})
    return {"rows": rows, "supported": supported, "total": total,
            "score": supported / total if total else 0.0}

This is the quote-match version of the faithfulness idea from 3.5: a claim is faithful when it can be traced to source text. If you have `ragas` installed, its Faithfulness metric automates the same idea with an LLM judge, breaking claims into atomic statements and checking each against the context. The quote-match version ships with the project because it has no extra dependencies and its verdicts are fully auditable. Your eval table should show both the score and the individual rows: a score of 0.9 with the one failure being a fabricated citation reads very differently from a score of 0.9 with one uncited claim.

## Deliverables

1. **Working code:** `research_team.py` plus a `docs/` folder with 3 to 6 real markdown sources. Runs end to end with no API key; LLM mode runs with `OPENAI_API_KEY` set.
2. **A sample report:** the final draft for one brief, with citations, saved as `sample_report.md`.
3. **An eval result table:** the faithfulness rows and score for the sample report, saved as `eval_results.md`. Include at least one run where the critic actually rejected a draft, so the table shows the loop doing its job.
4. **A README:** what the team does, the architecture choice and why (supervisor vs pipeline vs handoffs in two or three sentences), how to run it, the LLM swap instructions, the eval score with your honest reading of the failures, and the cost of one full run (token counts or "stub mode: zero").

## Marking rubric

| Criterion | Weight | What earns full marks |
|---|---|---|
| Functionality | 30% | Runs end to end without a key; `Send` fan-out actually parallelises researchers; critic loop respects the budget; approval gate pauses and resumes on the same thread. |
| Architecture choices justified | 20% | README explains the pipeline choice against supervisor and handoffs, the shared vs per-agent state split, and where persistence and the approval gate sit, each in concrete terms, not slogans. |
| Evals | 20% | Faithfulness table present with per-claim rows; at least one unsupported or uncited finding shown and discussed honestly; score not presented as the whole story. |
| Code quality | 15% | Reducer used for `findings`; no node writes a field it should only read; fan-out capped; interrupt payload JSON-serializable; LLM swaps keep the stub contracts. |
| README | 15% | A stranger can run it, understands the architecture decision, and learns the cost of one run and the known failure modes. |

## What can go wrong

- **The critic loop never converges.** The critic finds a nit, the writer fixes the nit and introduces another, forever. The budget is the fix, and the gate must route forward when the budget is spent, not raise. Log every rejection with the reason so you can see the loop's taste: a critic that rejects twice for the same reason is a critic with a bad rubric, not a writer with a bad draft.
- **Researcher context overload.** Give a researcher the whole document set "for context" and two things happen: it gets slower, and its quotes start blending sources, which the eval then flags as unsupported. The per-agent `Send` input is the discipline. One question, one file.
- **Citation fabrication.** The writer invents a plausible quote and cites a real file. The quote-match eval catches the verbatim case. A paraphrased fabrication needs the LLM-judge version (ragas Faithfulness sketch from 3.5). Say in your README which one you ran and what it cannot catch.
- **Unbounded fan-out.** A planner with no cap turns a broad brief into forty researchers. Cap `MAX_RESEARCHERS`, and make the cap a constant at the top of the file, not a number buried in the planner. Forty parallel model calls is also a rate-limit incident, not just a bill.
- **Shared-state write conflicts.** Two nodes appending to a plain list field instead of a reducer field silently drops one writer's output. The `Annotated[list, operator.add]` reducer is load-bearing. If a finding ever goes missing, check the reducer first.
- **The approval that approves nothing.** An `interrupt()` payload with the full draft and no summary trains reviewers to click approve without reading. Keep the payload small: the question, the budget used, a preview. The full report is one click away, not the default view.
- **Evaluating the stub and claiming the LLM.** The quote-match eval on stub output will score suspiciously well, because the stub copies text verbatim. That score measures the stub, not the system. Run the eval on the LLM-backed draft for the table you submit, and say so.

## Check your understanding

**1. Why does this project use a pipeline with one loop instead of a supervisor, and when would the supervisor be the better choice?**
The research workflow has a natural fixed order and exactly one decision point (accept or revise), so a pipeline is deterministic, testable, and cheap. A supervisor fits when the next step genuinely depends on what just happened in ways you cannot enumerate up front, for example a debugging agent that might need to re-plan, call different tools, or escalate at any point. Supervisor flexibility costs a routing model call per step and silent routing mistakes.

**2. The critic keeps rejecting the draft and the run has used its full revision budget. What should the gate do, and why not just raise an error or loop again?**
Route forward to the approval gate and ship the draft as-is. Looping again has no budget left, so it is either an infinite loop or an arbitrary extra lap, and raising leaves the run stuck with no artefact. The shipped draft carries its flaws, and the eval table records them, which is honest and debuggable. A stuck run is neither.

**3. Each researcher receives its sub-question and source file through `Send`, not through the shared state. Name two problems this prevents.**
First, context overload: a researcher with the full plan and all documents carries far more context than it needs, which costs tokens and causes cross-source quote blending. Second, accidental coupling: with only its own input, a researcher cannot read or depend on another researcher's output, so the map phase stays truly parallel and a failure in one researcher cannot corrupt another's input.

**4. Your faithfulness eval reports 0.95, but one of the failures is a fabricated citation. Why is the row-level table more informative than the score, and what would you do about the fabrication?**
The score averages away the severity: one fabricated citation is a trust-destroying failure, while one uncited claim is a minor gap, and 0.95 hides which one you have. The fix depends on the cause: if the writer paraphrased a real quote, tighten the writer prompt to require verbatim quotes; if it invented the quote, that is the failure the critic should have caught, so the critic's rubric needs a quote-verification step. Either way the README should name it.

**5. Where does the human approval go in this system, and why not inside the researcher loop or before the planner?**
After the critic accepts the draft and before publication, because publication is the only irreversible step. Approving inside the researcher loop is busywork (read-only source scans need no approval, and busywork gets rubber-stamped). Approving before the planner approves nothing concrete: there is no artefact to judge yet. One approval, at the one irreversible step, keeps the human's attention where it matters.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.